In [ ]:
# Cell 1: Environment Setup

include("scripts/om_helpers.jl")
include("scripts/dictionaries.jl")
using .OMHelpers
using .Dictionaries

using OMJulia

# --- Configuration ---

# 1. Define the directory containing your models
MODEL_DIR = abspath("Models")

# 2. Define the main model name 
MODEL = "ManyBESS"

# 3. Define the auxiliary model name
AUX_MODEL = "$(MODEL)_auxiliary"

# 4. Path to the Dynawo package.mo
DYNAWO_PKG_PATH   = "/.../dynawo/sources/Models/Modelica/Dynawo/package.mo"

# 4. Path to the Modelica package.mo
MODELICA_PKG_PATH = "/.../dynawo/OpenModelica/lib/omlibrary/Modelica/package.mo"

In [ ]:
# Cell 2 : Main script

# Start OMC and load libraries
omc = OMJulia.OMCSession()
sendExpression(omc, "loadFile(\"$MODELICA_PKG_PATH\")")
sendExpression(omc, "loadModel(Complex)")
sendExpression(omc, "loadModel(ModelicaServices)")
sendExpression(omc, "loadFile(\"$DYNAWO_PKG_PATH\")")

# Load the dynamic model
sendExpression(omc, "cd(\"$MODEL_DIR\")")
sendExpression(omc, "loadFile(\"$MODEL.mo\")")

# Copy original model into auxiliary model
sendExpression(omc, "copyClass($MODEL, \"$AUX_MODEL\")")

# Build component dictionary (from the original model)
components = get_all_components(omc, MODEL)

# Apply dictionary-driven replacements (infiniteBus, BESS)
apply_replacements!(omc, MODEL, AUX_MODEL, REPLACEMENTS, components)

# Delete connections to the sources
delete_source_connections!(omc, AUX_MODEL, components)

# Delete sources
delete_sources!(omc, AUX_MODEL, components)

# Add INIT models (dictionary-driven, here only BESS_INIT)
add_init_models!(omc, MODEL, AUX_MODEL, INIT_MODELS, components)

# Add Initial equations
add_init_equations!(omc, AUX_MODEL, components, INIT_MODELS)

# Save the model
aux_file = joinpath(MODEL_DIR, AUX_MODEL * ".mo")
sendExpression(omc, "saveModel(\"$aux_file\", $AUX_MODEL)")
patch_aux_equations!(aux_file)

# Re-load patched auxiliary .mo from disk and validate
sendExpression(omc, "deleteClass($AUX_MODEL)")
sendExpression(omc, "loadFile(\"$aux_file\")")
sendExpression(omc, "clearMessages()")
chk = sendExpression(omc, "checkModel($AUX_MODEL)", parsed=false)
println(chk)